# Aplicación de matrices de peso — Dataset Penguins (con normalización)

In [ ]:
import numpy as np
import seaborn as sns

## Matrices de peso y función de activación

In [ ]:
# W1
W1 = np.array([
    [ 0.3247309,  0.4846657],
    [ 6.7838711, -7.8951786],
    [ 3.9395028,  7.6781381],
    [-9.5005687, -3.7045610],
    [-8.3601396, -3.3254885]
])

# W2
W2 = np.array([
    [-5.235093,  0.22828501,   5.837634],
    [45.775386,  0.01075652, 115.631134],
    [95.253059, -0.01723614, -159.534197]
])

# W3
W3 = np.array([
    [ 0.487656773, -0.4724747,  0.294139023],
    [ 0.001744911,  1.0050892, -1.008197823],
    [ 0.921152682, -0.9609333,  1.287541532],
    [-1.003918311,  1.0044096, -0.004639103]
])

In [ ]:
# Función de activación sigmoide
def f_act(X):
    activada = np.array([1 / (1 + np.exp(-x)) for x in X], dtype=np.float64)
    return activada

## Carga y preparación de datos

In [ ]:
penguins = sns.load_dataset('penguins')
penguins.head()

In [ ]:
penguins.info()

In [ ]:
# Eliminar filas con valores nulos
penguins = penguins.dropna().reset_index(drop=True)
print(f'Filas después de dropna: {len(penguins)}')

In [ ]:
# Columnas de entrada
xcols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

# Especies en orden alfabético (igual que R)
especie = np.array(['Adelie', 'Chinstrap', 'Gentoo'])
print('Especies:', especie)

## Normalización (0 a 1)
La fórmula para la normalización es: `(x - min) / (max - min)`

In [ ]:
def normalizar(col):
    return (col - col.min()) / (col.max() - col.min())

X = penguins[xcols].copy()
X = X.apply(normalizar)

# Verificar que todo esté entre 0 y 1
X.describe()

In [ ]:
# Agregar columna bias = 1 en la primera posición
X.insert(0, 'bias', 1)
X.head()

## Predicción con la red neuronal

In [ ]:
prediccion = []

for index, fila in X.iterrows():
    # Capa oculta 1
    capa1 = f_act(fila.dot(W1))
    capa1 = np.insert(capa1, 0, 1)

    # Capa oculta 2
    capa2 = f_act(capa1.dot(W2))
    capa2 = np.insert(capa2, 0, 1)

    # Capa de salida
    salida = f_act(capa2.dot(W3))

    # La especie con mayor activación es la predicción
    prediccion.append(especie[np.argmax(salida)])

In [ ]:
# Agregar predicciones al dataframe original
penguins['Prediccion'] = prediccion
penguins.head(10)

## Evaluación de resultados

In [ ]:
# Filas predichas incorrectamente
erroneas = penguins[penguins['species'] != penguins['Prediccion']]
print(f'Predicciones erróneas: {len(erroneas)} de {len(penguins)}')
erroneas

In [ ]:
# Exactitud del modelo
correctas = (penguins['species'] == penguins['Prediccion']).sum()
exactitud = correctas / len(penguins) * 100
print(f'Exactitud: {exactitud:.2f}%')